# Call-level caching

cash caches a notebook **statement**. When the statement is a cheap wrapper
around an expensive call, the statement alone is the wrong unit:

| statement | statement-level caching alone | why |
|---|---|---|
| `out.append(compute(x))` | reuses nothing | the append changes an object that already exists, so there is no result to store |
| `s += compute(x)` | reuses only an unchanged *prefix* | each iteration reads `s`, so its key includes every iteration before it: reorder the list and the tail runs again |

So cash also caches the expensive `compute(x)` call itself, one level below the
statement, with no directive needed. The cheap wrapper still runs every time
(the append really happens); the expensive call is served from the cache.
`# @cash:no-cache-calls` turns this off for a statement or a whole cell.

**How to use this notebook:** run the cells top to bottom once, then re-run
single cells as each section says. Judge by the badge, not the clock: the clock
cannot tell "ran again" from "restored but slow".

In [ ]:
import cash
%cash_on

## Setup

`compute()` sleeps for a second and returns. It is deliberately **pure**: it
writes nothing outside itself.

That is not just tidiness, it is the precondition for section 2. Reordering a
loop is free because a call cache keys on **arguments**, not on execution
history. A callee that writes to a global smuggles execution history back in:
the write has to be reproduced in the new order, so the call cannot be served
and the reorder costs real executions again. Cash stays correct either way —
you just pay for the side effect.

> **How to read the evidence.** Not the wall clock, which cannot tell
> "recomputed" from "restored but slow". And not a counter incremented inside
> `compute()`: such a counter is itself a side effect, so cash restores it on a
> hit and it reads the same whether the call ran or was served. Open the
> statement's badge row and read the `sub-call compute(x): 0/1 hit` line
> underneath it. `0/1` is a real execution, `1/1` is a served one.

In [ ]:
import time

def compute(x):
    time.sleep(1.0)      # stand-in for real work
    return x + 1

def merge(a, b):
    return a + b


---
## 1. An append loop

The list is created in **its own cell**, and that matters. When `out = []` is
in the *same* cell as the loop, cash can cache the whole cell as one unit. The
interesting, and more common, case is a container built earlier and appended to
later.

Run the next cell once, then re-run **only the loop cell after it**, twice.

The append really runs each time, so `out` grows as it would without cash.
Each loop row in the badge reads EXECUTED. The reuse shows in the line under
it: `sub-call compute(x): 0/1 hit` on the first run, `1/1 hit` after that.
Three seconds of sleeping, gone.

In [ ]:
out = []          # built HERE, appended to in the next cell

In [ ]:
for x in [1, 2, 3]:
    out.append(compute(x))


### Turning it off with `# @cash:no-cache-calls`

Same loop, one comment added. Run the `out2 = []` cell once, then re-run the
loop cell **twice**.

Every run takes three seconds: with call-level caching off for this statement,
`compute(x)` really runs each time. The badge shows no `sub-call` line and no
`compute() [intercepted]` row.

In [ ]:
out2 = []         # again, built in its own cell

In [ ]:
# @cash:no-cache-calls
for x in [1, 2, 3]:
    out2.append(compute(x))

print("out2 =", out2)


---
## 2. Reordering a loop

Run the next cell once. Then **change the list to `[30, 20, 10]`** and run it
again.

The accumulator `s` makes each iteration's *statement* depend on every
iteration before it, so a reorder misses the statement cache and the loop runs
again. But `compute(x)` is cached by its arguments, not by what ran before it,
so every `sub-call` line reads `1/1 hit`: the reorder costs nothing.

> **This holds because `compute()` is pure.** Give it a side effect (append to
> a global, write a file) and the effect has to happen again in the new order,
> so the call runs again and the reorder costs full price. That is cash being
> correct rather than fast. If a reorder unexpectedly costs executions, look
> for a write escaping the callee.

In [ ]:
s = 0
for x in [10, 20, 30]:      # <-- try [30, 20, 10] on the second run
    s += compute(x)

print("SUM", s)


### The same fold, with `# @cash:no-cache-calls`

Run once, then **reorder the list any way you like** and run again.

With call-level caching off for this statement, both the statement entries and
the calls miss on a reorder: every `compute(x)` in the new order runs again.

In [ ]:
s2 = 0
# @cash:no-cache-calls
for x in [11, 22, 33]:      # <-- reorder freely; every compute() call re-runs
    s2 += compute(x)

print("SUM", s2)


---
## 3. What is (and isn't) eligible

A call is only cached on its own when it does **not** read the statement's own
target. If it does, the call *is* the fold, and there is no order-independent
value to pull out of it.

| statement | cached call |
|---|---|
| `s += compute(x)` | `compute(x)` |
| `out.append(compute(x))` | `compute(x)` |
| `prices[k] = compute(k)` | `compute(k)` |
| `s = merge(s, x)` | none: the call reads `s` |
| `df.sort_values(inplace=True)` | none: the mutation *is* the work |

The next cell is the last-but-one row: `merge` reads `acc`, the statement's own
target, so there is nothing to extract. cash does not warn about this; most
statements have no eligible call, and that is normal. The badge simply has no
`merge()` row, unlike the `compute() [intercepted]` rows above.

In [ ]:
acc = 0
acc = merge(acc, 5)      # `merge` reads `acc`, the target -> nothing to extract; not intercepted

print("acc =", acc)

---
## What to look for

- **The badge's `sub-call compute(x): n/1 hit` line** tells you whether the
  work happened. A counter written inside the callee does not: it is a side
  effect, so cash restores it on a hit and it reads the same either way.
- **The tag `compute() [intercepted]`** means cash wrapped that call itself.
  Plain `@cache` (no tag) means you decorated the function yourself; no row at
  all means nothing was wrapped or decorated.
- **No warning when nothing is eligible.** Check the badge, not your warnings.

### What is never intercepted

- **Already-decorated functions.** They are cached already; wrapping them again
  would split their hits across two cache entries.
- **Builtins.** A hot loop must not pay for a cache key per `len()`.
- **Bound methods** (`model.predict(x)`). Caching a method puts `self` in the
  key, which needs your judgement; decorate the method yourself when you want
  that.

### Reference

- [Call-level caching, and `# @cash:no-cache-calls`](https://cash-lib.readthedocs.io/en/latest/annotations/#call-level-caching-default-and-cashno-cache-calls)
- [Reordering a loop's items](https://cash-lib.readthedocs.io/en/latest/known-limitations/#reordering-a-loops-items-re-runs-the-tail)

> **Placement matters.** A `@cash:` directive applies to the statement *below*
> it. On a loop, put it on the line above the `for` header. `no-cache-calls`
> (like `no-cache`) at the very top of a cell applies to every top-level
> statement in that cell. See
> [the annotations reference](https://cash-lib.readthedocs.io/en/latest/annotations/#a-cell-header-opt-out-does-not-reach-past-an-intervening-statement).